In [6]:
import pandas as pd
import sys
import re
import serial
import serial.tools.list_ports as serialport
import plotly.figure_factory as ff
import plotly.graph_objs as go

import subprocess

In [7]:
# sample data:
data = [{'value': 10, 'onset': 1.0, 'offset': 2.0, 'duration': 1.0, 'occurence': 1},
        {'value': 20, 'onset': 4.0, 'offset': 4.1, 'duration': 0.1, 'occurence': 1},
        {'value': 30, 'onset': 6.0, 'offset': 9.0, 'duration': 3.0, 'occurence': 1},
        {'value': 40, 'onset': 9.0, 'offset': 11.0, 'duration': 2.0, 'occurence': 1},
        {'value': 10, 'onset': 12.0, 'offset': 14.0, 'duration': 2.0, 'occurence': 2}
]

data_frame = pd.DataFrame(data= data, columns = ['value', 'onset', 'offset', 'duration', 'occurence'])

summary = data_frame
summary = summary[['value', 'occurence']]
summary = summary.drop_duplicates(subset = ['value'], keep = 'last')

print("The data frame:\n", data_frame, "\n\n\n The summaries table:\n", summary)

The data frame:
    value  onset  offset  duration  occurence
0     10    1.0     2.0       1.0          1
1     20    4.0     4.1       0.1          1
2     30    6.0     9.0       3.0          1
3     40    9.0    11.0       2.0          1
4     10   12.0    14.0       2.0          2 


 The summaries table:
    value  occurence
1     20          1
2     30          1
3     40          1
4     10          2


In [8]:
# Initialize a figure:
fig = ff.create_table(data_frame)

# temp = data_frame

# # Graph data:
# values = temp[['value']]
# times = temp[['duration']]    # TODO: gotta figure this out


# # Make a trace for the graph plot:
# fig.add_trace(go.Scatter(x=times, y=values,
#                     marker=dict(color='#0099ff'),
#                     name='Marker Value',
#                     xaxis='x2', yaxis='y2'))
                    

# fig.update_layout(
#     title_text = 'Markers plotted as digital signals',
#     margin = {'t':50, 'b':100},
#     xaxis = {'domain': [0, .5]},
#     xaxis2 = {'domain': [0.6, 1.]},
#     yaxis2 = {'anchor': 'x2', 'title': 'Marker Value'}
# )

# Plot the figure:
fig.show()

In [9]:
def test_serial(dev_name = ''):
    """ returns open serial ports (COM_portnumber) identified by string matching with desired_ports[]

        :raises EnvironmentError:
            On unsupported or unknown platforms
        :returns:
            A list of the serial ports available on the system
    """
    # check windows OS:
    if sys.platform.startswith('win'):
        com_ports = serialport.comports()
    else:
        raise EnvironmentError('Unsupported platform')

    device_name = []
    property = []
    device_number = 0
    counter = []   # for detecting chosen device connected or not
    desired_ports = ['USB Serial Device', 'Ardruino', 'Lombardo']   # specify the desired device descriptions here to choose the desired device

    for port in com_ports:
        try:
            if (desired_ports[0] or desired_ports[1] or desired_ports[2]) in port.description:
                device_number += 1
                # Initialize the COM port:
                s = serial.Serial(port.device)
                if s.isOpen():
                    if dev_name != '':
                        if s._port_handle != dev_name:
                            counter.append(1)
                    print(s.portstr)
                    device_name.append(s.port)
                    property.append(port.hwid)    # define any desirable property instead of hwid
                    s.close()
        except (OSError, serial.SerialException):
            pass
    
    print("Found", device_number, "UsbParMar devices.")
    if not counter:
        pass
    else:
        raise Exception("Connected device(s) did not match the requested device")

    print(property)

    return device_name


print("Available COM ports: ", test_serial())

COM7
COM8
Found 2 UsbParMar devices.
['USB VID:PID=2341:8036 SER=7 LOCATION=1-1.1:x.0', 'USB VID:PID=2341:8036 SER=7 LOCATION=1-1.2:x.0']
Available COM ports:  ['COM7', 'COM8']


In [10]:
AllocatedResources = subprocess.Popen("wmic path Win32_PNPAllocatedResource Get", \
                    shell=True, stdout=subprocess.PIPE).stdout.read().splitlines()

PNPDeviceIDstrings = subprocess.Popen("wmic path Win32_ParallelPort Get PNPDeviceID /VALUE", \
                               shell=True, stdout=subprocess.PIPE).stdout.read()

PNPDeviceIDs = []
for line in PNPDeviceIDstrings.splitlines():
    if line != b'':
        PNPDeviceID = str(line.strip())
        PNPDeviceID = PNPDeviceID.replace('\'','')
        PNPDeviceID = PNPDeviceID.replace('\\\\','\\')
        PNPDeviceID = PNPDeviceID.replace('&amp;','&')
        PNPDeviceID = PNPDeviceID.split('=')[1]
        PNPDeviceID = PNPDeviceID.encode()
        PNPDeviceIDs.append(PNPDeviceID)

# list available LPT port addresses
AddressesFound = []
for r in AllocatedResources:
    for PNPDeviceID in PNPDeviceIDs:
        if r.find(PNPDeviceID) != -1:
            StartingAddresses = re.findall('StartingAddress="[\d]+"', r.decode())
            HexStartingAddress = hex(int(re.findall('[\d]+', StartingAddresses[0])[0]))
            AddressesFound.append(HexStartingAddress[2:].upper())

NumAddressesFound = len(AddressesFound)
print(NumAddressesFound)
print(AddressesFound)

# check port address
if len(sys.argv) == 1: # no port specified to check
    print("False")
    print("Warning: Missing argument. No port specified to check.")
else:
    if sys.argv[1] in AddressesFound:
        print("True") # Port is a correct port
    else:
        print("False")

print("Found " + str(int(NumAddressesFound/2)) + " LPT address(es): ")
for i in range(0,NumAddressesFound,2):
    print(AddressesFound[i]) 

0
[]
False
Found 0 LPT address(es): 
